# EPI Recorder Investor Demo (`v2.8.3`)

This Colab demo is designed for investors and shows EPI's strongest current capabilities in a short, reliable flow.

What it demonstrates:
- real `.epi` artifact creation
- embedded `policy.json`
- embedded `analysis.json`
- human review stored as `review.json`
- tamper detection
- current offline viewer rendered inside the notebook

**Run all cells from top to bottom.**

In [ ]:
# @title 1. Install EPI Recorder { display-mode: "form" }
INSTALL_SOURCE = "pypi"  # @param ["pypi", "github"]
GITHUB_REF = "main"

import subprocess
import sys

if INSTALL_SOURCE == "pypi":
    install_target = "epi-recorder==2.8.3"
else:
    install_target = f"git+https://github.com/mohdibrahimaiml/epi-recorder.git@{GITHUB_REF}"

cmd = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--force-reinstall",
    "--no-cache-dir",
    "cryptography<44",
    "rich<14",
    "pydantic<=2.12.3",
    "jedi>=0.16",
    install_target,
]
result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode != 0:
    print(result.stdout)
    print(result.stderr)
    raise RuntimeError("Dependency install failed")

import epi_core
print(f"[OK] Installed epi-recorder {epi_core.__version__}")
print("This demo focuses on fault intelligence and evidence trust, not generic tracing.")

In [ ]:
# @title 2. Create Policy and Generate the Main .epi Artifact { display-mode: "form" }
import json
import shutil
from pathlib import Path

workspace = Path("/content/epi_investor_demo")
workspace.mkdir(parents=True, exist_ok=True)

for stale in [
    workspace / "investor_demo.epi",
    workspace / "investor_demo_reviewed.epi",
    workspace / "investor_demo_tampered.epi",
    workspace / "investor_demo.py",
    workspace / "epi_policy.json",
]:
    if stale.exists():
        if stale.is_dir():
            shutil.rmtree(stale)
        else:
            stale.unlink()

policy = {
    "system_name": "loan-underwriting-agent",
    "system_version": "2.8.3-demo",
    "policy_version": "2026-03-18",
    "profile_id": "finance.loan-underwriting",
    "rules": [
        {
            "id": "R001",
            "name": "Do Not Exceed Balance",
            "severity": "critical",
            "description": "The agent must not approve an amount above the known balance.",
            "type": "constraint_guard",
            "watch_for": ["balance", "available_balance", "limit"],
            "violation_if": "approved_amount > balance"
        },
        {
            "id": "R002",
            "name": "Verify Identity Before Refund",
            "severity": "high",
            "description": "Identity verification must happen before refund.",
            "type": "sequence_guard",
            "required_before": "refund",
            "must_call": "verify_identity"
        },
        {
            "id": "R003",
            "name": "Human Approval Above 10000",
            "severity": "high",
            "description": "Amounts above 10,000 require human approval.",
            "type": "threshold_guard",
            "watch_for": ["amount", "approved_amount", "loan_amount"],
            "threshold_value": 10000,
            "required_action": "human_approval"
        },
        {
            "id": "R004",
            "name": "Never Output Secrets",
            "severity": "critical",
            "description": "The system must not expose secret-like tokens in output.",
            "type": "prohibition_guard",
            "pattern": "sk-[A-Za-z0-9]+"
        }
    ]
}

(workspace / "epi_policy.json").write_text(json.dumps(policy, indent=2), encoding="utf-8")

agent_code = r'''
from pathlib import Path
from epi_recorder import record

output = Path("/content/epi_investor_demo/investor_demo.epi")

with record(
    str(output),
    workflow_name="Investor Demo Financial Fault Workflow",
    goal="Show where an AI-assisted financial workflow went wrong",
    notes="Intentional policy and heuristic violations for investor demo",
    metadata_tags=["investor-demo", "finance", "fault-analysis"],
) as epi:
    epi.log_step("tool.response", {
        "tool": "fetch_account_context",
        "account_id": "ACC-4401",
        "customer_id": "CUST-9001",
        "balance": 500.0,
        "currency": "USD",
    })

    epi.log_step("llm.error", {
        "error": "Rate limit exceeded while fetching sanctions data",
        "provider": "demo-llm",
    })

    epi.log_step("tool.call", {
        "tool": "approve_loan_disbursement",
        "amount": 2000.0,
        "account_id": "ACC-4401",
        "reason": "urgent disbursement",
    })

    epi.log_step("tool.call", {
        "tool": "process_refund",
        "amount": 200.0,
        "reference": "REF-2026-001",
    })

    epi.log_step("tool.call", {
        "tool": "approve_wire_transfer",
        "amount": 15000.0,
        "destination": "vendor-settlement",
    })

    epi.log_step("llm.response", {
        "text": "Escalation note: temporary key observed as sk-ABC123SECRET for retry flow.",
        "model": "demo-llm",
    })

    epi.log_step("tool.response", {
        "tool": "create_case",
        "case_reference": "CASE-7702",
        "status": "pending_manual_review",
    })

print(f"Created artifact: {output}")
'''

(workspace / "investor_demo.py").write_text(agent_code, encoding="utf-8")

%cd /content/epi_investor_demo
!python investor_demo.py

epi_file = workspace / "investor_demo.epi"
assert epi_file.exists(), "Expected investor_demo.epi to be created"
print(f"\n[PACKAGE] {epi_file.name} ({epi_file.stat().st_size / 1024:.0f} KB)")

In [ ]:
# @title 3. Verify, Inspect Faults, and Append Human Review { display-mode: "form" }
import json
import shutil
import sys
import zipfile
from pathlib import Path

from epi_cli.keys import KeyManager, generate_default_keypair_if_missing
from epi_core.review import ReviewRecord, add_review_to_artifact, make_review_entry

epi_file = Path("/content/epi_investor_demo/investor_demo.epi")
assert epi_file.exists(), "Run cell 2 first"

print("=" * 70)
print("VERIFY ARTIFACT")
!{sys.executable} -m epi_cli.main verify {epi_file}

print("\n" + "=" * 70)
print("FAULT ANALYZER OUTPUT")
!{sys.executable} -m epi_cli.main analyze {epi_file}

with zipfile.ZipFile(epi_file, "r") as zf:
    policy_json = json.loads(zf.read("policy.json").decode("utf-8"))
    analysis_json = json.loads(zf.read("analysis.json").decode("utf-8"))

print("\nEmbedded contents:")
print("- policy rules:", len(policy_json.get("rules", [])))
print("- primary fault:", (analysis_json.get("primary_fault") or {}).get("fault_type"))
print("- violated rule:", (analysis_json.get("primary_fault") or {}).get("rule_id"))
print("- why it matters:", (analysis_json.get("primary_fault") or {}).get("why_it_matters"))

reviewed = Path("/content/epi_investor_demo/investor_demo_reviewed.epi")
if reviewed.exists():
    reviewed.unlink()
shutil.copy2(epi_file, reviewed)

generate_default_keypair_if_missing(console_output=False)
review = ReviewRecord(
    reviewed_by="investor.demo.reviewer@epilabs.org",
    reviews=[
        make_review_entry(
            fault=analysis_json["primary_fault"],
            outcome="confirmed_fault",
            notes="Confirmed for demo: this violation was intentionally introduced to show EPI review.",
            reviewer="investor.demo.reviewer@epilabs.org",
        )
    ],
)
review.sign(KeyManager().load_private_key("default"))
add_review_to_artifact(reviewed, review)
assert reviewed.exists(), "Expected reviewed artifact to be created"
print("\n[REVIEW] Added review.json to investor_demo_reviewed.epi")

In [ ]:
# @title 4. Tamper Test - Forge the Reviewed Evidence { display-mode: "form" }
import json
import sys
import zipfile
from pathlib import Path

source_epi = Path("/content/epi_investor_demo/investor_demo_reviewed.epi")
tampered = Path("/content/epi_investor_demo/investor_demo_tampered.epi")
assert source_epi.exists(), "Run cell 3 first"
if tampered.exists():
    tampered.unlink()

with zipfile.ZipFile(source_epi, "r") as src, zipfile.ZipFile(tampered, "w", compression=zipfile.ZIP_DEFLATED) as dst:
    for item in src.infolist():
        data = src.read(item.filename)
        if item.filename == "analysis.json":
            payload = json.loads(data.decode("utf-8"))
            payload["fault_detected"] = False
            payload.setdefault("summary", {})
            payload["summary"]["headline"] = "FORGED: analysis changed after sealing"
            data = json.dumps(payload, indent=2).encode("utf-8")
        dst.writestr(item, data)

print("=" * 60)
print("SIGNED REVIEWED ARTIFACT:")
!{sys.executable} -m epi_cli.main verify {source_epi}
print("\n" + "=" * 60)
print("FORGED ARTIFACT:")
!{sys.executable} -m epi_cli.main verify {tampered}
print("=" * 60)
assert tampered.exists(), "Expected tampered artifact to be created"
print("Cell 5 defaults to the forged artifact so the viewer shows the trust break inside the notebook.")

In [ ]:
# @title 5. Open the Current EPI Viewer and Download the Artifact { display-mode: "form" }
VIEW_TARGET = "tampered"  # @param ["reviewed", "tampered", "generated"]

import html
import json
import re
import tempfile
import zipfile
from pathlib import Path

from IPython.display import HTML, display
from epi_cli.view import _build_viewer_context
from epi_core.container import EPIContainer
from epi_core.schemas import ManifestModel

generated_epi = Path("/content/epi_investor_demo/investor_demo.epi")
reviewed_epi = Path("/content/epi_investor_demo/investor_demo_reviewed.epi")
tampered_epi = Path("/content/epi_investor_demo/investor_demo_tampered.epi")
targets = {
    "generated": generated_epi,
    "reviewed": reviewed_epi,
    "tampered": tampered_epi,
}
epi_file = targets[VIEW_TARGET]
assert generated_epi.exists(), "Run cell 2 first"
assert epi_file.exists(), f"Missing {epi_file.name}; run the earlier cells first"

with zipfile.ZipFile(epi_file) as z:
    names = set(z.namelist())
    manifest = json.loads(z.read("manifest.json"))
    steps_raw = z.read("steps.jsonl").decode().strip()
    steps = [json.loads(line) for line in steps_raw.split("\n") if line]
    analysis = json.loads(z.read("analysis.json").decode()) if "analysis.json" in names else None
    policy = json.loads(z.read("policy.json").decode()) if "policy.json" in names else None
    review = json.loads(z.read("review.json").decode()) if "review.json" in names else None

if analysis is not None:
    analysis["fault_detected"] = bool(analysis.get("primary_fault") or analysis.get("fault_detected"))
    analysis.setdefault("summary", {})
    if not analysis["summary"].get("headline") and analysis.get("primary_fault"):
        pf = analysis["primary_fault"]
        analysis["summary"]["headline"] = pf.get("plain_english") or "Primary fault detected"
    analysis["disclaimer"] = "Policy violations are deterministic rule matches. Heuristic observations are pattern-based and should be reviewed by a human."

with tempfile.TemporaryDirectory() as tmpdir:
    source_dir = Path(tmpdir)
    if steps_raw:
        (source_dir / "steps.jsonl").write_text(steps_raw + ("\n" if not steps_raw.endswith("\n") else ""), encoding="utf-8")
    else:
        (source_dir / "steps.jsonl").write_text("", encoding="utf-8")
    if analysis is not None:
        (source_dir / "analysis.json").write_text(json.dumps(analysis, indent=2), encoding="utf-8")
    if policy is not None:
        (source_dir / "policy.json").write_text(json.dumps(policy, indent=2), encoding="utf-8")
    if review is not None:
        (source_dir / "review.json").write_text(json.dumps(review, indent=2), encoding="utf-8")
    viewer = EPIContainer._create_embedded_viewer(source_dir, ManifestModel.model_validate(manifest))

view_context = json.dumps(_build_viewer_context(epi_file)).replace("</", "<\\/")
if 'id="epi-view-context"' in viewer:
    viewer = re.sub(
        r'<script id="epi-view-context" type="application/json">.*?</script>',
        f'<script id="epi-view-context" type="application/json">{view_context}</script>',
        viewer,
        flags=re.DOTALL,
    )
elif "</head>" in viewer:
    viewer = viewer.replace("</head>", f'<script id="epi-view-context" type="application/json">{view_context}</script></head>')
else:
    viewer = f'<script id="epi-view-context" type="application/json">{view_context}</script>' + viewer

escaped = html.escape(viewer)
trust = _build_viewer_context(epi_file)
trust_label = "Tampered" if not trust.get("integrity_ok") else ("Signed" if trust.get("signature_valid") else "Unsigned")
banner = "#dc2626" if trust_label == "Tampered" else ("#10b981" if trust_label == "Signed" else "#d97706")

display(HTML(f'''
<div style="border:2px solid {banner}; border-radius:8px; overflow:hidden; margin:10px 0;">
  <div style="background:{banner}; color:white; padding:10px 16px; font-weight:bold;">
    EPI Viewer ({VIEW_TARGET}) - {epi_file.name} - Trust: {trust_label}
  </div>
  <iframe srcdoc="{escaped}" width="100%" height="700" style="border:none;" sandbox="allow-scripts allow-same-origin"></iframe>
</div>
'''))

print("Viewer file:", epi_file.name)
print("Steps captured:", len(steps))
print("Primary fault:", ((analysis or {}).get("primary_fault") or {}).get("fault_type", "Unavailable"))
print("Trust state:", trust_label)
print("Default viewer target is now the forged artifact so the trust break is obvious.")
print("Change VIEW_TARGET to 'reviewed' if you want to show the signed reviewed artifact instead.")
print("Default download: the main generated artifact investor_demo.epi")

try:
    from google.colab import files
    files.download(str(generated_epi))
except Exception:
    pass
